In [1]:
import gmsh
import dassflow2d as df2d

## Exemple de génération d'un cas plus complexe qu'un simple channel

### 1- Génération de la géométrie et du maillage avec GMSH - cas avec deux entrées et deux sorties

In [ ]:
gmsh.initialize()
gmsh.model.add("carre_complexe_H")
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)

lc = 2.0 

p1  = gmsh.model.geo.addPoint(0, 0, 0, lc)   # Bas Gauche (Ext)
p2  = gmsh.model.geo.addPoint(10, 0, 0, lc)  # Bas Gauche (Int)
p3  = gmsh.model.geo.addPoint(10, 20, 0, lc) # Pont Bas Gauche
p4  = gmsh.model.geo.addPoint(40, 20, 0, lc) # Pont Bas Droite
p5  = gmsh.model.geo.addPoint(40, 0, 0, lc)  # Bas Droite (Int)
p6  = gmsh.model.geo.addPoint(50, 0, 0, lc)  # Bas Droite (Ext)
p7  = gmsh.model.geo.addPoint(50, 50, 0, lc) # Haut Droite (Ext)
p8  = gmsh.model.geo.addPoint(40, 50, 0, lc) # Haut Droite (Int)
p9  = gmsh.model.geo.addPoint(40, 30, 0, lc) # Pont Haut Droite
p10 = gmsh.model.geo.addPoint(10, 30, 0, lc) # Pont Haut Gauche
p11 = gmsh.model.geo.addPoint(10, 50, 0, lc) # Haut Gauche (Int)
p12 = gmsh.model.geo.addPoint(0, 50, 0, lc)  # Haut Gauche (Ext)


l_inlet1 = gmsh.model.geo.addLine(p1, p2)
l_w1 = gmsh.model.geo.addLine(p2, p3)
l_w2 = gmsh.model.geo.addLine(p3, p4)
l_w3 = gmsh.model.geo.addLine(p4, p5)
l_outlet1 = gmsh.model.geo.addLine(p5, p6)
l_w4 = gmsh.model.geo.addLine(p6, p7)
l_outlet2 = gmsh.model.geo.addLine(p7, p8)
l_w5 = gmsh.model.geo.addLine(p8, p9)
l_w6 = gmsh.model.geo.addLine(p9, p10)
l_w7 = gmsh.model.geo.addLine(p10, p11)
l_inlet2 = gmsh.model.geo.addLine(p11, p12)
l_w8 = gmsh.model.geo.addLine(p12, p1)

# Surface
cl = gmsh.model.geo.addCurveLoop([
    l_inlet1, l_w1, l_w2, l_w3, 
    l_outlet1, l_w4, l_outlet2, 
    l_w5, l_w6, l_w7, l_inlet2, l_w8
])
s = gmsh.model.geo.addPlaneSurface([cl])

gmsh.model.geo.synchronize()

# --- 1. DEFINITION DU DOMAINE ---

gmsh.model.addPhysicalGroup(2, [s], tag=1, name="DOMAIN")

# --- 2. CONDITIONS AUX LIMITES ---

gmsh.model.addPhysicalGroup(1, [l_inlet1], tag=10, name="INLET:discharg1_1") # Deux inflow différents
gmsh.model.addPhysicalGroup(1, [l_outlet1], tag=20, name="INLET:discharg1_2")

gmsh.model.addPhysicalGroup(1, [l_inlet2], tag=11, name="OUTLET:hpresc_1") # Deux sorties avec des hauteurs différentes
gmsh.model.addPhysicalGroup(1, [l_outlet2], tag=21, name="OUTLET:hpresc_2")

# --- 3. GENERATION ---

gmsh.model.mesh.generate(2)

# Optionnel : Essayer de faire des quads (carrés) au lieu de triangles
gmsh.model.mesh.recombine() 

gmsh.write("modele_H.msh")
gmsh.fltk.run()
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.00069132s, CPU 0.000813s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0100546s, CPU 0.010215s)
Info    : 470 nodes 950 elements
Info    : Recombining 2D mesh...
Info    : Blossom: 1127 internal 140 closed
Info    : Blossom recombination completed (Wall 0.00946099s, CPU 0.009959s): 390 quads, 0 triangles, 0 invalid quads, 0 quads with Q < 0.1, avg Q = 0.817166, min Q = 0.419232
Info    : D

X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x0


### 2- Convertion du maillage au format Dassflow2D

In [3]:
from GMSH_to_df2d_librairy import convert_gmsh_to_df2d

Input_mesh = "modele_H.msh" # Doit être dans le dossier courant
folder_output="modele_H_bin"

#---------------------------------------- #
# Visualisation du maillage avec Gmsh
gmsh.initialize()
gmsh.open(Input_mesh)
gmsh.fltk.run()
gmsh.finalize()
#---------------------------------------- #

print(f"=== CONVERSION Test for {Input_mesh} ===")
convert_gmsh_to_df2d(Input_MSH=Input_mesh, Sim_duration=14400, Default_Q=10.0, Default_H=1.0, Default_Z=1.0, folder_output=folder_output)
print(f"=== CONVERSION Terminee ===")

Info    : Reading 'modele_H.msh'...
Info    : 461 nodes
Info    : 410 elements
Info    : Done reading 'modele_H.msh'
-------------------------------------------------------
Version       : 4.14.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 20250902
Build host    : gmsh.info
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blas[petsc] Blossom Cgns DIntegration Dlopen DomHex Eigen[contrib] Fltk Gmm[contrib] Hxt Jpeg Kbipack Lapack[petsc] LinuxJoystick MathEx[contrib] Med Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom PETSc Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.11
PETSc version : 3.14.4 (real arithmtic)
OCC version   : 7.8.1
MED version   : 4.1.0
Packaged by   : geuzaine
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info

X_ChangeProperty: BadValue (integer parameter out of range for operation) 0x0


=== CONVERSION Test for modele_H.msh ===
--- 1. Chargement du maillage modele_H.msh ---
Info    : Reading 'modele_H.msh'...
Info    : 461 nodes
Info    : 410 elements
Info    : Done reading 'modele_H.msh'
--- 2. Analyse des conditions aux limites ---
--- 3. Ecriture de modele_H_bin/modele_H.geo ---
--- 4. Ecriture de modele_H_bin/bc.txt ---
--- 5. Génération data (avec variantes) ---
 -> Création de modele_H_bin/land_uses.txt (Format Fortran corrigé)...
 -> Création de modele_H_bin/hydrograph.txt avec 2 courbe(s)...
 -> Création de modele_H_bin/hpresc.txt avec 2 courbe(s)...
--- TERMINE AVEC SUCCES ---
=== CONVERSION Terminee ===


### 3- Modification de la bathymétrie et des conditions limites

In [4]:
from Update_files_tools import apply_bathymetry, update_boundary_files
import os
# -------------------------------------------------------------------------- #
#           EXEMPLE D'UTILISATION DE LA FONCTION apply_bathymetry
# -------------------------------------------------------------------------- #
geo_file = os.path.join(folder_output, "modele_H.geo")

def f(x, y):
    return (-0.5 * y) + 10

print(f"--- Application de la nouvelle bathymétrie depuis la fonction f(x,y) ---")
apply_bathymetry(f, geo_file)
print(f"--- Bathymétrie appliquée et fichier {geo_file} mis à jour ---")

# ---------------------------------------------------------------------------------- #
#           EXEMPLE D'UTILISATION DE  LA FONCTION update_boundary_files
# ---------------------------------------------------------------------------------- #


times = [0, 3600, 7200, 10800, 14400]  # en secondes
flows = [0.0, 5.0, 150.0, 30.0, 0.0]      # en m3/s

print(f"--- Mise à jour de l'hydrographe avec les nouvelles valeurs ---")
bc_hydro_file = os.path.join(folder_output, "hydrograph.txt")
update_boundary_files(bc_hydro_file, target_group_id=1, times=times, values=flows)

flows = [0.0, 10.0, 250.0, 40.0, 5.0]      # en m3/s
update_boundary_files(bc_hydro_file, target_group_id=2, times=times, values=flows)
print(f"--- Hydrographe {bc_hydro_file} mis à jour ---")

hpresc_values = [0.0, 5.0, 30.0, 20.0, 15.0]      # en mètres
print(f"--- Mise à jour de hpresc avec les nouvelles valeurs ---")
bc_hpresc_file = os.path.join(folder_output, "hpresc.txt")
update_boundary_files(bc_hpresc_file, target_group_id=1, times=times, values=hpresc_values)

hpresc_values = [0.0, 10.0, 50.0, 30.0, 20.0] 
update_boundary_files(bc_hpresc_file, target_group_id=2, times=times, values=hpresc_values)
print(f"--- hpresc {bc_hpresc_file} mis à jour ---")

# ---------------------------------------------------------------------------------- #

--- Application de la nouvelle bathymétrie depuis la fonction f(x,y) ---
Chargement du maillage modele_H_bin/modele_H.geo...
Application de la bathymétrie...
Fichier mis à jour avec succès.
--- Bathymétrie appliquée et fichier modele_H_bin/modele_H.geo mis à jour ---
--- Mise à jour de l'hydrographe avec les nouvelles valeurs ---
 -> Mise à jour du bloc n°1 (sur 2 blocs présents)...
Succès : Le fichier modele_H_bin/hydrograph.txt a été mis à jour.
 -> Mise à jour du bloc n°2 (sur 2 blocs présents)...
Succès : Le fichier modele_H_bin/hydrograph.txt a été mis à jour.
--- Hydrographe modele_H_bin/hydrograph.txt mis à jour ---
--- Mise à jour de hpresc avec les nouvelles valeurs ---
 -> Mise à jour du bloc n°1 (sur 2 blocs présents)...
Succès : Le fichier modele_H_bin/hpresc.txt a été mis à jour.
 -> Mise à jour du bloc n°2 (sur 2 blocs présents)...
Succès : Le fichier modele_H_bin/hpresc.txt a été mis à jour.
--- hpresc modele_H_bin/hpresc.txt mis à jour ---


### 4- Lancement d'une simulation avec Dassflow2D (version master)

In [2]:

import matplotlib.pyplot as plt
import shutil, os
import numpy as np
import csv
from mpi4py import MPI

# ------------------------------------------ Avec version dassflow2d master ------------------------------------------ #

# TODO : adapt this into ? model.meshing object + mesh_extent_box from a real dassflow mesh/setup (with wrapped object)
class mesh_box:
    def __init__(self, xmin, ymax, ncol, nrow, dx, dy, x_cell, y_cell):
        self.xmin = xmin
        self.ymax = ymax
        self.ncol = ncol
        self.nrow = nrow 
        self.dx = dx
        self.dy = dy
        self.x_cell = x_cell
        self.y_cell = y_cell


##############################################################################################################

dassflow_dir="/home/dgomez/dassflow2d_master"
code_dir =  f"{dassflow_dir}/code"
bin_dir = f"{os.getcwd()}/modele_H_bin"

##########
# Initialise bin
##########

os.chdir(bin_dir)

##########
# MPI
##########

df2d.wrapping.m_mpi.init_mpi()
rank = df2d.wrapping.m_mpi.get_proc() # get the rank and number of processors
nproc = df2d.wrapping.m_mpi.get_np()
mpi = [rank, nproc]

##########
# Model
##########

df2d.wrapping.read_input(f"{bin_dir}/input.txt") 

my_model = df2d.dassflowmodel(bin_dir =  bin_dir, hdf5_path = f"{bin_dir}/res/simu.hdf5" , run_type = "direct", clean = True)

my_model.config.set({"ts": "10040"})
my_model.config.set({"dtw": "1000"})
my_model.config.set({"dt": "10"})
                    
my_model.init_all()

#my_model.meshing.plot()

plotter = my_model.boundary.plot(what="meshing", notebook=False) # for a local run remove notebook option or set notebook=False


h_initiale = 0.01
my_model.kernel.dof.h[:] = h_initiale
my_model.kernel.dof0.h[:] = h_initiale

my_model.run()


################################################################################################################
# Results observations
################################################################################################################

plotter = my_model.outputs.result.plot_field(my_mesh = my_model.meshing.mesh_pyvista,
                                             what = "bathy",
                                             title_plot = "Bathymetry elevation",
                                             notebook = False)


u = my_model.outputs.result.u
v = my_model.outputs.result.v
h = my_model.outputs.result.h


plotter = my_model.outputs.result.plot_field(my_mesh = my_model.meshing.mesh_pyvista,
                                             what = "h",
                                             when = 0,
                                             title_scale_bar ="h [m] ",
                                             title_plot = f"Water depth at time = {my_model.outputs.result.all_time[0]}  s "  ,
                                             notebook=False)



plotter = my_model.outputs.result.plot_field(my_mesh = my_model.meshing.mesh_pyvista,
                                             what = "h",
                                             when = -1,
                                             title_scale_bar ="h [m] ",
                                             title_plot = f"Water depth at time = {my_model.outputs.result.all_time[-1]}  s "  ,
                                             notebook=False)

os.chdir("..")


clean= True
 ERROR in file input.txt, see below for more information on the error.                                                                                                                                                                                                                                                                                                                                                                                                                                                           
None
{'mesh_name': 'modele_H.geo', 'ts': 4400.0, 'dta': 0, 'dtw': 1000.0, 'dtp': 10.0, 'dt': 10.0, 'temp_scheme': 'euler', 'spatial_scheme': 'first_b1', 'adapt_dt': 1, 'cfl': 0.8, 'feedback_inflow': 1, 'coef_feedback': 0.8, 'heps': 0, 'friction': 1, 'g': 10.0, 'w_tecplot': 1, 'w_vtk': 1, 'w_gnuplot': 1, 'w_obs': 0, 'use_obs': 0, 'max_nt_for_adjoint': 2500, 'c_manning': 0, 'c_manning_beta': 0, 'c_bathy': 0, 'c_hydrograph': 0, 'c_ratcurve': 0, 'c_rain': 0, 'c_ic': 0, 'res